# 07b_lock — 설정 선택과 결정 규칙 고정 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 2.5·4.3·4.5절. 03~07a의 dev 점수로 다음을 정하고 `lock/selection-lock.json`에 고정한다. **test 결과는 읽지 않는다.**

- 원천마다 단계별 isotonic 보정(어댑터). 설정 선택용은 dev 5분할 교차 보정, test에 쓸 것은 dev 전체로 맞춘다.
- 원천별 단계 간 실패 상관 ρ, 비용 목표(원천별 고정 시작 비용 범위의 25·50·75%), (원천, 규칙, κ, 비용 수준)마다 λ.
- 단계 비용은 Claude 1:3:5가 주 기준이다. 민감도로 두 가지 비용의 목표와 λ도 함께 만든다(설정 선택은 1:3:5로만): 집단 모델의 평균 데이터 가격, 길이 반영(입력 + 5 × 출력 토큰 중앙값, `length-inputs/` 필요).
- 평가 제외: 영어가 아닌 문항 246건과 문서 제공형(RAG형) SQuAD·BoolQ 180건(id 목록)을 dev와 보정 출력에서 지운다. 학습은 그대로다.
- 방법마다 설정 하나: 상향 고려 규칙의 "고정 시작 단계 대비 이득"을 원천·κ·비용 수준에 걸쳐 평균한 값이 가장 큰 설정.
- RT-08(v2 재학습)과 운영 R0(v1)의 보정·λ(A 분할, 집단 기준·대표 모델 기준).

04, 05a, 05b, 06, 07a가 모두 끝난 뒤 실행한다. 빠진 점수가 있으면 멈춘다.

**여기서 멈추고 lock 요약(`lock/lock-summary.md`)을 보고한다.** 확인을 받은 뒤에 08_test를 연다.

**런타임: CPU면 충분하다.** 10분 안팎.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-lock.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.6.1', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
FINAL = RUN / 'lock'
assert not FINAL.exists(), '이미 lock이 있다. 다시 만들려면 폴더를 다른 이름으로 옮긴 뒤 실행'

def run_logged(script, args):
    """Run a script and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final' / script), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

run_logged('lock.py', ['--run-dir', RUN, '--repo', REPO, '--output-dir', FINAL])
print((FINAL / 'lock-summary.md').read_text())